# Chapter 4 — Conversational Agents

Aegis has been handed clean, structured alerts. Real security work starts with a human
typing a paragraph. Somebody has to turn that into a structured incident record.

Two jobs, and they are different: **extraction** (parse what they already told you) and
**elicitation** (ask only for what is missing). Conflate them and you build the bot that
asks for information the user just gave it.




## Setup

This lab installs from **one** `requirements.txt`.




In [1]:
REPO_URL = "https://github.com/gstripling00/ai-engineer.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


repo: /content/aegis


In [2]:
# --no-warn-conflicts silences a cosmetic Colab-only notice about `requests`;
# see the comment block at the top of requirements.txt. Real resolver errors still raise.
!pip -q install --no-warn-conflicts -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 695.4/695.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

Now verify the environment before running any lab code. This is the same check CI
runs, and it catches the one dependency conflict that would otherwise waste your
afternoon. It also confirms this chapter's source folder is in the checkout.


In [3]:
!python tools/check_env.py --chapter 4

dependencies
  ok      langgraph              open-source agent track (StateGraph/END)
  ok      langchain-core         message and tool primitives
  ok      langchain-community    RAGAS dependency — see the pin note
  ok      google-adk             Google Cloud agent track (Agent, Workflow)
  ok      mcp                    tool discovery and hardening (Ch 3, 9, 11)
  ok      openai                 the default real-model tier
  ok      langchain-openai       wires OpenAI into RAGAS
  ok      ragas                  evaluation (Ch 10)
  ok      sacrebleu              required by RAGAS BleuScore
  ok      opentelemetry-sdk      tracing (Ch 10)
  ok      chromadb               vector store (Ch 6)
  ok      rank-bm25              sparse retrieval for hybrid search (Ch 6)
  ok      pytest                 the test suite

critical pin
  ok      langchain-community 0.3.29 (compatible with ragas)

model access
  absent  OPENAI_API_KEY not set
          Offline labs still run: AEGIS_MODEL=mock
  

### Choosing a model tier

The labs read `AEGIS_MODEL` and swap the model behind a single seam:

| Tier | Cost | Determinism | Use it for |
|---|---|---|---|
| `mock` | free, no key | identical every run | learning the control flow; the test suite; CI |
| `openai` | billed per call | varies run to run | seeing a real model make these decisions |

Start on `mock`. Everything in this chapter runs there. When you switch to
`openai`, the code does not change — only the seam does.

Set the key from the environment, never as a literal in a cell. In Colab use the
key icon in the sidebar (Secrets); the cell below reads it without printing it.


In [4]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key

# To use a real model instead, uncomment these two lines:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: "); os.environ["AEGIS_MODEL"] = "openai"

print("model tier:", os.environ["AEGIS_MODEL"])


model tier: mock


## Intent recognition, and the multi-intent trap

Before slot filling you must know which workflow the message belongs to. The trap is
returning **one** label: real messages carry more than one intent, and the dropped half
is often the urgent half.


In [5]:
import sys
sys.path.insert(0, "labs/chapter-04-conversational-agents-slot-filling")   # this chapter's source lives beside the notebook
from intake.intent import classify_intent, split_multi_intent

message = "I got a phishing email and also my VPN is broken"
verdict = classify_intent(message)

print("message:", message)
print("multi_intent:", verdict["multi_intent"])
print("intents found:", list(verdict["all_intents"]))
print()
for action in split_multi_intent(message):
    print(f'  {action["intent"]:18} -> {action["action"]:9} cues={action["cues"]}')
print()
print("The helpdesk half is OUT OF SCOPE - so it is handed off, not dropped.")
print("Silently ignoring half a request is how users learn not to trust the agent.")


message: I got a phishing email and also my VPN is broken
multi_intent: True
intents found: ['report_phishing', 'it_helpdesk']

  report_phishing    -> handle    cues=['phish']
  it_helpdesk        -> hand_off  cues=['vpn', 'broken']

The helpdesk half is OUT OF SCOPE - so it is handed off, not dropped.
Silently ignoring half a request is how users learn not to trust the agent.


## Entity extraction

The first message usually contains most of the form. Note the defanged-URL pattern
(`hxxp://`): security people write links that way on purpose, and an extractor that does
not know it misses the most important field in the report.

Note also `clean()` — stripping trailing punctuation. An indicator captured as
`hxxp://bad[.]example.` will never match a blocklist or cluster into a Chapter 5
campaign. That is an indicator-quality bug that passes any test checking only presence.


In [6]:
from intake.slot_filling import extract_slots, REQUIRED_SLOTS
from common import soc

found = extract_slots(soc.PHISHING_REPORT)
for slot, value in found.items():
    print(f'{slot:15} {value!r}')
print()
print(f'{len(found)} of {len(REQUIRED_SLOTS)} slots filled from the first message alone')
print("no trailing punctuation on any value - check the URL and the time")


report_type     'suspicious_email'
sender          'helpdesk@it-support-reset.example'
malicious_url   'hxxp://it-support-reset[.]example'
clicked         'no'
approx_time     '8:40am'

5 of 5 slots filled from the first message alone
no trailing punctuation on any value - check the URL and the time


## Slot filling: elicit the gap, not the form

Extraction runs on **every** message, then the agent asks for the first thing still
missing. A complete report therefore costs zero questions.


In [7]:
from intake.slot_filling import IntakeState, apply_extraction

state = IntakeState()
state = apply_extraction(state, soc.PHISHING_REPORT)

print("questions asked:", len(state.missing()))
print("complete:", state.complete())
print()
vague = apply_extraction(IntakeState(), "I think I got a phishing email this morning")
print("vague report still missing:", vague.missing())
print("next question:", vague.next_question())


questions asked: 0
complete: True

vague report still missing: ['sender', 'malicious_url', 'clicked', 'approx_time']
next question: What was the sender's email address?


## Grounded extraction

A pattern extractor can only *miss*. A model gains coverage — and gains the ability to
**invent a sender address into a security incident record**.

The rule: a model may propose; only the text may confirm.


In [8]:
def grounded_extract(text: str, proposed: dict) -> dict:
    low = text.lower()
    accepted, rejected = {}, {}
    for slot, value in proposed.items():
        if isinstance(value, str) and "@" in value and value.lower() not in low:
            rejected[slot] = value
        else:
            accepted[slot] = value
    return {"accepted": accepted, "rejected": rejected}


vague_report = "I got a suspicious email this morning"
model_proposed = {"report_type": "suspicious_email",
                  "sender": "ceo@yourcompany.example"}    # never appeared in the text

result = grounded_extract(vague_report, model_proposed)
print("accepted:", result["accepted"])
print("rejected:", result["rejected"])
print()
print("Without grounding that address enters an incident record,")
print("and someone's CEO is investigated for a report they never sent.")


accepted: {'report_type': 'suspicious_email'}
rejected: {'sender': 'ceo@yourcompany.example'}

Without grounding that address enters an incident record,
and someone's CEO is investigated for a report they never sent.


## Interruptions

People do not answer the question you asked. Mid-interview they change the subject. An
agent that treats every message as an answer writes *"wait, am I compromised?"* into
the sender field.

The fix is not cleverness — it is keeping the intake state so the interview can be
paused and **resumed** rather than restarted.


In [9]:
from intake.intent import is_interruption, handle_turn

state = apply_extraction(IntakeState(), "I think I got a phishing email")
pending = state.next_question()
print("agent asked:", pending)

for message in ["wait, am I already compromised?",
                "it was from helpdesk@it-support-reset.example"]:
    turn = handle_turn(state, message, apply_extraction,
                       lambda q: "Not yet - no sign of access on that account.")
    state = turn["state"]
    print()
    print(f'user: {message}')
    print(f'  detected as: {turn["kind"]}')
    if turn["kind"] == "interruption":
        print(f'  answered, then resumed to: {turn["resumed_question"]}')
        print(f'  same question as before: {turn["resumed_question"] == pending}')
    else:
        print(f'  slot captured; next question: {turn["reply"]}')


agent asked: What was the sender's email address?

user: wait, am I already compromised?
  detected as: interruption
  answered, then resumed to: What was the sender's email address?
  same question as before: True

user: it was from helpdesk@it-support-reset.example
  detected as: answer
  slot captured; next question: What URL were you asked to visit? (paste it defanged if you like)


---

## What you built

An intake agent that recognizes multiple intents, hands off what it cannot do, extracts
before it asks, refuses to accept a slot the text does not support, and survives an
interruption without restarting the interview.

**Next:** Chapter 5 gives Aegis memory, and the third phishing report from this sender
stops looking like an isolated incident.
